# S11 · Squash many columns down to two

Real data usually has many columns, and you cannot draw a dot in five or fifty
dimensions. **PCA** squashes the many columns down to just two, keeping as much of
the real spread as possible, so you can plot the data on a flat chart and see the
groups. That is exactly what we need before clustering: a way to *look* at wide data.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press play on each cell, top
  to bottom, and read the plain-English note above each one.
- Curious what a "covariance matrix" or "eigenvector" is? You do not need them for
  the main line. When they appear, the primer `primers/vectors_and_matrices.md` has
  the picture, and there is an optional **Stretch** cell that goes into the maths.
- Already confident? Skip ahead to the cells marked **Stretch (optional)**.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy, matplotlib and scikit-learn.
# Google Colab already ships all three, so there is nothing to install.
print("Setup complete - nothing to install.")

These are all the tools this notebook needs, imported once up front.

In [ ]:
import numpy as np                              # fast maths on lists of numbers
import matplotlib.pyplot as plt                  # drawing charts
from sklearn.datasets import load_iris           # a small 4-column dataset that ships with sklearn
from sklearn.preprocessing import StandardScaler # puts every column on the same scale
from sklearn.decomposition import PCA            # the squash-the-columns tool
from sklearn.manifold import TSNE                # a second way to make a 2-D picture

## Step 1 — load some data with too many columns to plot

The iris flower dataset has 4 measurements per flower: sepal length, sepal width,
petal length, petal width. Four columns is already too many to plot directly (a flat
chart only has two axes), so it is a perfect case for squashing down to two.

In [ ]:
# Set a seed so anything random is the same for everyone.
np.random.seed(0)

iris = load_iris()
flower_measurements = iris.data        # shape (150, 4): 150 flowers, 4 columns
flower_species = iris.target           # 0, 1 or 2: the species of each flower

print("data shape:", flower_measurements.shape)
print("the 4 columns:", iris.feature_names)
print("first 3 flowers:")
print(flower_measurements[:3])

## Step 2 — scale the columns first

PCA works with distances and spread, so columns measured in different ranges would
not get a fair say. We **standardise** each column to have mean 0 and a spread of 1,
so every measurement counts equally. This is the same "scale first" rule that
clustering needs, and it is easy to forget.

In [ ]:
scaler = StandardScaler()
scaled_measurements = scaler.fit_transform(flower_measurements)

# After scaling, every column should have mean about 0 and spread (std) about 1.
print("mean of each column after scaling:", scaled_measurements.mean(axis=0).round(2))
print("std  of each column after scaling:", scaled_measurements.std(axis=0).round(2))

## Step 3 — run PCA

PCA finds new directions through the data, called **principal components**, ordered
by how much of the spread they capture. We ask for all 4 at first so we can see how
much each one is worth. The first component is the single direction along which the
flowers differ the most.

In [ ]:
# n_components=4 keeps all 4 directions for now (the data has 4 columns).
pca = PCA(n_components=4)
pca.fit(scaled_measurements)

# How much of the total spread each component captures, as a fraction.
variance_explained = pca.explained_variance_ratio_
print("share of the spread captured by each component:")
print(variance_explained.round(3))
print()
print("share kept by just the first 2 components:",
      round(variance_explained[0] + variance_explained[1], 3))

## Step 4 — the scree plot

A **scree plot** shows how much of the spread each component captures. The first
components carry the most; later ones carry little. We also draw the running total,
so we can see how much we keep if we stop after two components.

In [ ]:
component_numbers = range(1, 5)
cumulative_variance = np.cumsum(variance_explained)

plt.figure(figsize=(7, 5))

# Bars: share of the spread captured by each single component.
plt.bar(list(component_numbers), variance_explained,
        color="#9DC3E6", label="each component")

# Line: the running total as we add components.
plt.plot(list(component_numbers), cumulative_variance,
         "-o", color="#C0392B", label="running total")

plt.xlabel("principal component")
plt.ylabel("share of the spread captured")
plt.title("Scree plot: the first 2 components carry most of the spread")
plt.legend()
plt.show()

## Step 5 — squash to two columns and plot

Now we keep just the first 2 components and use them as new x and y coordinates. This
turns each 4-number flower into a single dot on a flat plot. We colour by the true
species only to check the picture. PCA itself never saw the species; it worked from
the measurements alone.

In [ ]:
# Keep only the top 2 components and rewrite the data in those 2 coordinates.
pca_2d = PCA(n_components=2)
flowers_in_2d = pca_2d.fit_transform(scaled_measurements)

print("data shape after PCA:", flowers_in_2d.shape, "(2 columns now)")

plt.figure(figsize=(7, 5))
plt.scatter(flowers_in_2d[:, 0], flowers_in_2d[:, 1],
            c=flower_species, cmap="viridis", s=30)
plt.xlabel("principal component 1")
plt.ylabel("principal component 2")
plt.title("Iris flowers squashed onto their first 2 principal components")
plt.show()

print("The species separate into clumps - PCA found that structure on its own.")

## Step 6 (optional) — a quick t-SNE picture

**t-SNE** is another way to squeeze many columns into 2 for viewing. Unlike PCA it is
non-linear and tries hard to keep similar points close together, which often makes
the groups pop out. Use it to *see* whether groups exist, but do not read meaning
into the distances or the empty space between the blobs. Those are not reliable.

In [ ]:
# perplexity roughly controls how many neighbours each point pays attention to.
tsne = TSNE(n_components=2, perplexity=30, random_state=0)
flowers_tsne = tsne.fit_transform(scaled_measurements)

plt.figure(figsize=(7, 5))
plt.scatter(flowers_tsne[:, 0], flowers_tsne[:, 1],
            c=flower_species, cmap="viridis", s=30)
plt.xlabel("t-SNE axis 1 (no real units)")
plt.ylabel("t-SNE axis 2 (no real units)")
plt.title("t-SNE view of the same flowers - good for spotting groups")
plt.show()

### Stretch (optional) — what PCA is really doing

Skip this unless you have met matrices and eigenvectors, or you are curious. Here is
the satisfying part. The principal components are not magic; they are exactly the
**eigenvectors of the covariance matrix** of the scaled data, and the spread each one
captures is its **eigenvalue**. The covariance matrix is a small table saying how the
columns vary together. We build it and eigen-decompose it by hand with `numpy`.

In [ ]:
# The covariance matrix: how the 4 columns vary together. It is 4x4.
# rowvar=False tells numpy that each column (not each row) is a variable.
covariance_matrix = np.cov(scaled_measurements, rowvar=False)
print("covariance matrix shape:", covariance_matrix.shape)

# Eigen-decomposition. We use eigh because a covariance matrix is symmetric.
eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)

# eigh returns them smallest-first, so we reverse to put the largest first.
eigenvalues = eigenvalues[::-1]
print("eigenvalues, largest first:", eigenvalues.round(3))

### Stretch (optional) — check it matches scikit-learn

If PCA really is the eigen-decomposition, the eigenvalues we just computed by hand
should equal the spread `scikit-learn` reported for each component. Let us compare
them side by side.

In [ ]:
# scikit-learn stores the spread along each component in explained_variance_.
print("our eigenvalues       :", eigenvalues.round(3))
print("sklearn PCA variances :", pca.explained_variance_.round(3))
print()
print("They match - PCA is the eigen-decomposition of the covariance matrix.")

## What you just did

You took 4-column data and squashed it down to 2 columns you could plot, keeping most
of the spread, and the species fell into visible clumps. That is the tool we need
before clustering: a way to *see* wide data. And if you did the stretch cells, you
saw that PCA is not a new piece of maths at all, just the eigen-decomposition of the
covariance matrix, doing a useful new job.

Next notebook: `03_customer_segmentation.ipynb`, the lab, where clustering and PCA
come together on a real business task.